Step 1a - Indexing (Document Ingestion)

In [ ]:
pip install yt-dlp

In [ ]:
import yt_dlp
import json

def get_transcript_ytdlp(video_id: str, lang: str = "en") -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    ydl_opts = {
        "writesubtitles": True,
        "writeautomaticsub": True,
        "subtitleslangs": [lang],
        "subtitlesformat": "json3",
        "skip_download": True,
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        subs = info.get("subtitles", {}) or info.get("automatic_captions", {})
        if lang not in subs:
            available = list(subs.keys())
            raise ValueError(f"No '{lang}' transcript. Available: {available}")
        sub_url = next(
            (s["url"] for s in subs[lang] if s.get("ext") == "json3"),
            None
        )
        if not sub_url:
            raise ValueError("json3 format not available for this video.")
    import urllib.request
    with urllib.request.urlopen(sub_url) as response:
        data = json.loads(response.read())
    lines = []
    for event in data.get("events", []):
        for seg in event.get("segs", []):
            text = seg.get("utf8", "").strip()
            if text and text != "\n":
                lines.append(text)
    return " ".join(lines)

# ✅ Call directly — no if __name__ block in notebooks
transcript = get_transcript_ytdlp("Gfr50f6ZBvo")
print(len(transcript))
print(transcript[:500])

133836
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


Step 1b - Indexing (Text Splitting)

In [ ]:
!pip install langchain_text_splitters

In [ ]:
pip install -U langchain-openai

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

168

In [ ]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
!pip install faiss-cpu

In [ ]:
pip install langchain langchain-community langchain-openai faiss-cpu

In [ ]:
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Get API key
openai_api_key = userdata.get("OPENAI_API_KEY")

# Create embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=openai_api_key
)

# Create vector store from documents
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [ ]:
vector_store.index_to_docstore_id

{0: '7948787e-f3ca-477a-abd6-e22d50e53c85',
 1: '4b5cdd11-7511-4496-916e-dae56624c6cb',
 2: '60a2cfe7-d99b-43fd-accd-06d17423d523',
 3: 'dcdf43a5-76e4-4888-9df8-ed3ba8864304',
 4: '3f3af13c-adad-4088-8611-fb64d05513ee',
 5: '83b0d439-bfba-47fc-a2fe-5e420bcca9e7',
 6: '82e41559-b5e9-4461-a70e-04d59b775c8c',
 7: 'ff743c41-1054-47ec-9030-69356295c931',
 8: '7d5b617c-1329-4864-a87c-2e7b84631186',
 9: '266af7cc-fbba-4731-9b8f-1f856cb47c03',
 10: '66e13e8e-563e-47ec-ae48-c195de519f94',
 11: 'a6b532cd-c7d1-46d2-be04-777381964420',
 12: '141470a0-253c-4f8d-84cc-3246f114783c',
 13: '15c72cd9-2e0f-42a1-904c-ee59b782a480',
 14: 'a0396a6c-727c-463f-84a5-0a49ac79c09c',
 15: '878eaed5-24c5-4db1-8fa5-47729e5e1362',
 16: 'c39b2e45-e2cf-4102-8034-dd81c597aedc',
 17: 'cebdbcc1-b8d3-4af9-aeaf-d6dadd779ce6',
 18: '3fa290af-816f-4acd-b108-d3d7615f5218',
 19: '14877750-aa19-4bcd-a5e0-aa13f8a52d94',
 20: 'bb00b59b-6050-41ff-8f5d-c633ed31678f',
 21: 'c877092c-c10c-4c7e-8e8a-8fd735a014c9',
 22: 'bcab3798-202c-

In [ ]:
vector_store.get_by_ids(['5f140039-aa9d-42ad-8c72-df48d94414ef'])

[]

Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7e02eaa97980>, search_kwargs={'k': 3})

In [ ]:
retriever.invoke('Who is Demus Hasabis?')

[Document(id='7948787e-f3ca-477a-abd6-e22d50e53c85', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

Step 3 - Augmentation

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

# Get API key from Colab secrets
openai_api_key = userdata.get("OPENAI_API_KEY")

# Create model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key
)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
      You are a helpful assistant.
      Answer only from the provided transcript context.
      If the context is insufficient, just say I don't know

      {context}
      Question: {question}
    """,
    input_variables=["context", "question"]
)

In [ ]:
question = "is the topic of aliens discussed in this video? if yes then what?"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='fc425ced-7304-4f8a-bc9e-459e0aa14482', metadata={}, page_content="other option of course we could enhance ourselves and and without devices we we are already sort of symbiotic with our compute devices right with our phones and other things and you know this stuff like neural link and etc that could be could could advance that further um so i think there's lots of lots of really amazing possibilities uh that i could foresee from here well let me ask you some wild questions so out there looking for friends do you think there's a lot of alien civilizations out there so i guess this also goes back to your origin of life question too because i think that that's key um my personal opinion looking at all this and and you know it's one of my hobbies physics i guess so so i i you know it's something i think about a lot and talk to a lot of experts on and and and read a lot of books on and i think my feeling currently is that that we are alone i think that's the most likely scenari

In [ ]:
# in the relevant chunks that we got, we have chunks that are separated. so combine all the chunks together
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

"\n      You are a helpful assistant.\n      Answer only from the provided transcript context.\n      If the context is insufficient, just say I don't know\n\n      other option of course we could enhance ourselves and and without devices we we are already sort of symbiotic with our compute devices right with our phones and other things and you know this stuff like neural link and etc that could be could could advance that further um so i think there's lots of lots of really amazing possibilities uh that i could foresee from here well let me ask you some wild questions so out there looking for friends do you think there's a lot of alien civilizations out there so i guess this also goes back to your origin of life question too because i think that that's key um my personal opinion looking at all this and and you know it's one of my hobbies physics i guess so so i i you know it's something i think about a lot and talk to a lot of experts on and and and read a lot of books on and i think 

Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

content='Yes, the topic of aliens is discussed in this video. The speaker expresses their personal opinion that we are likely alone in the universe, reasoning that we should have detected signs of alien civilizations if they existed, given our advances in technology. They mention that we have searched enough for evidence of other life forms but have found nothing, which leads them to be skeptical of the arguments suggesting that we may not be looking in the right way or that we are a primitive species not yet capable of space-faring. The speaker also discusses the variety that alien civilizations could encompass, suggesting there would be a range of characteristics among them, rather than a uniformity.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 639, 'total_tokens': 768, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_

**Building a Chain**

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('Who is Demis?')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm |parser

In [ ]:
main_chain.invoke("Can you summarize the video?")

'The video features a conversation about exploring deeper and potentially simpler explanations for fundamental concepts in physics, including consciousness, life, and gravity, as well as discussing the challenges of solving the mystery of intelligence at DeepMind. The participants express the idea that the standard model of physics is insufficient and advocate for fundamental changes in understanding. They also contemplate the interplay of science, engineering, algorithms, data, hardware, software, and human interaction in achieving breakthroughs in artificial intelligence. Overall, the conversation emphasizes the complexity of these topics and the honor of engaging in such discussions.'